# FinReq AI — Fine-Tuning & Evaluation on Google Colab

This notebook fine-tunes **Qwen2.5-7B-Instruct** (or 3B) using **Unsloth + QLoRA** and evaluates it on `test.jsonl`.

### Instructions for Files (No Google Drive Needed):
1. On the left sidebar of Google Colab, click the 📁 **Files** icon.
2. Drag and drop these 3 files from your PC into the left panel:
   - `train.jsonl` (from `C:\Users\vikas\Desktop\projectfinal\ml\dataset\data\train.jsonl`)
   - `val.jsonl` (from `C:\Users\vikas\Desktop\projectfinal\ml\dataset\data\val.jsonl`)
   - `test.jsonl` (from `C:\Users\vikas\Desktop\projectfinal\ml\dataset\data\test.jsonl`)
3. Run the cells below one by one.

### Step 1: Verify Uploaded Files

In [ ]:
import os

required_files = ['train.jsonl', 'val.jsonl', 'test.jsonl']
for fname in required_files:
    if not os.path.exists(fname):
        raise FileNotFoundError(f'Missing {fname}! Please drag and drop {fname} into the left Files panel.')
    size_kb = os.path.getsize(fname) / 1024
    print(f'✅ Found {fname} ({size_kb:.1f} KB)')

print('All 3 dataset files verified successfully!')

### Step 2: Install Unsloth and Dependencies

In [ ]:
%%capture
!pip install --no-deps unsloth
!pip install "torch" "transformers" "datasets" "accelerate" "bitsandbytes" "trl" "peft" "xformers"
print('Dependencies installed!')

### Step 3: Load Base Model in 4-bit (Runs on Free T4 GPU)

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
DTYPE = None  # None for auto-detection
LOAD_IN_4BIT = True

# Primary: unsloth/Qwen2.5-7B-Instruct-bnb-4bit (Fallback: unsloth/Qwen2.5-3B-Instruct-bnb-4bit)
MODEL_NAME = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)
print('Base model loaded successfully.')

### Step 4: Configure LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('LoRA adapter initialized.')

### Step 5: Format Datasets (Train, Val, Test)

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files={
    'train': 'train.jsonl',
    'val': 'val.jsonl',
    'test': 'test.jsonl'
})

def formatting_prompts_func(examples):
    convos = examples['messages']
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {'text': texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f'Train size: {len(dataset["train"])}')
print(f'Val size:   {len(dataset["val"])}')
print(f'Test size:  {len(dataset["test"])}')

### Step 6: Fine-Tune with SFTTrainer (Loss on Assistant Tokens Only)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

OUTPUT_DIR = '/content/outputs'

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['val'],
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_ratio=0.03,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=50,
        save_strategy='steps',
        save_steps=100,
        optim='paged_adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir=OUTPUT_DIR,
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)

print('Starting training...')
trainer_stats = trainer.train()
print('Training complete!')

### Step 7: Evaluate on Test Set (`test.jsonl`)

In [ ]:
# Fast inference mode for evaluation
FastLanguageModel.for_inference(model)

import json

# Evaluate on 10 held-out test examples
test_samples = dataset['test'].select(range(min(10, len(dataset['test']))))
valid_json_count = 0

print('--- Running Evaluation on Held-Out Test Samples ---')
for idx, sample in enumerate(test_samples):
    messages = sample['messages'][:2]  # system + user
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt'
    ).to('cuda')
    
    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=512,
        temperature=0.1,
        use_cache=True
    )
    
    response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
    
    # Check JSON validity
    is_json = False
    try:
        # Strip markdown fences if present
        clean_resp = response.strip()
        if clean_resp.startswith('```json'):
            clean_resp = clean_resp[7:-3].strip()
        elif clean_resp.startswith('```'):
            clean_resp = clean_resp[3:-3].strip()
        parsed = json.loads(clean_resp)
        is_json = True
        valid_json_count += 1
    except Exception:
        pass
    
    print(f'Test Sample #{idx+1} ({sample["task"]}): JSON Valid = {is_json}')
    if idx == 0:
        print(f'Sample Output preview:\n{response[:250]}...\n')

print(f'Test JSON Validity Score: {valid_json_count} / {len(test_samples)} ({valid_json_count/len(test_samples)*100:.1f}%)')

### Step 8: Export to GGUF (Q4_K_M)

In [ ]:
# Export directly into GGUF Q4_K_M for local Ollama CPU inference
model.save_pretrained_gguf('/content', tokenizer, quantization_method='q4_k_m')
print('Exported to GGUF in /content!')
!ls -lh /content/*.gguf

### Step 9: Download GGUF to Your Computer

In [ ]:
import glob
from google.colab import files

gguf_files = glob.glob('/content/*q4_k_m*.gguf') or glob.glob('/content/*.gguf')
if gguf_files:
    target = gguf_files[0]
    print(f'Starting browser download for: {target}')
    files.download(target)
else:
    print('Right click on the .gguf file in the left Files panel and click Download.')